In [3]:
"""
重新处理 Soho 餐饮 POI —— 修复业态过滤缺失的问题

背景：现有的 soho_food_pois_r1900.csv 是直接从未经业态过滤的
soho_food_pois_wide.csv 按半径截取得到的，混入了约28.6%的非食品
业态（零售/酒店/学校等），因为 environment.py 里 type_weight.map()
用了 fillna(1.0) 兜底，这些非食品记录被当成默认权重1.0的POI纳入了模型。

本脚本对宽域文件重新走一遍完整流程：业态过滤 → dark kitchen地址
排查 → 人工排除非消费端商户 → 按 r=1900m 半径截取 → 存新文件。

产出文件不覆盖旧文件，先另存为 _clean 版本，方便你对比新旧差异后
再决定要不要正式替换 environment.py 默认读取的文件名。
"""

import pandas as pd
import math

# ────────────────────────────────────────────────────────────
# 0. 路径与参数 —— 按你的实际情况改
# ────────────────────────────────────────────────────────────
DATA_DIR = "/Users/wusiyi/Documents/CASA/dis/DATA"
WIDE_POI_PATH = f"{DATA_DIR}/soho_food_pois_wide.csv"          # 输入：未过滤的宽域pull
OUTPUT_PATH   = f"{DATA_DIR}/soho_food_pois_r1900_clean.csv"    # 输出：过滤后的新文件

# 与 roadnetwork.ipynb cell 13 拉取宽域数据时用的同一个圆心，
# 保证半径截取的参照点和当初拉数据时一致
CENTER_LAT, CENTER_LON = 51.51350000000001, -0.1365
RADIUS_M = 1900.0

# ────────────────────────────────────────────────────────────
# 1. 读入宽域数据
# ────────────────────────────────────────────────────────────
df = pd.read_csv(WIDE_POI_PATH)
print(f"宽域原始记录数: {len(df)}")
print(df["BusinessType"].value_counts())
print()

# ────────────────────────────────────────────────────────────
# 2. 业态过滤 —— 只保留三类食品相关业态
#    （对应正文4.2.2节权重: 1.5 takeaway / 1.0 restaurant-cafe / 0.6 other）
# ────────────────────────────────────────────────────────────
food_types = ["Restaurant/Cafe/Canteen", "Takeaway/sandwich shop", "Other catering premises"]
df_food = df[df["BusinessType"].isin(food_types)].copy()

print(f"业态过滤后记录数: {len(df_food)}")
print(df_food["BusinessType"].value_counts())
print()

# 3. 【调整后的新第3步】先按 r=1900m 半径截取
# ────────────────────────────────────────────────────────────
def haversine(lat0, lon0, lat, lon):
    R = 6371000.0
    lat0r = math.radians(lat0)
    dlat = math.radians(lat - lat0)
    dlon = math.radians(lon - lon0)
    a = (math.sin(dlat / 2) ** 2
         + math.cos(lat0r) * math.cos(math.radians(lat)) * math.sin(dlon / 2) ** 2)
    return 2 * R * math.asin(math.sqrt(min(1.0, a)))

lon_col = next(c for c in df_food.columns if c.lower() in ("lon", "longitude"))
lat_col = next(c for c in df_food.columns if c.lower() in ("lat", "latitude"))

dists = [
    haversine(CENTER_LAT, CENTER_LON, lat, lon)
    for lat, lon in zip(df_food[lat_col], df_food[lon_col])
]
df_food_r1900 = df_food.loc[[d <= RADIUS_M for d in dists]].reset_index(drop=True)

print(f"半径筛选后（Soho r=1900m范围内、业态过滤后）记录数: {len(df_food_r1900)}")
print(df_food_r1900["BusinessType"].value_counts())
print()

# ────────────────────────────────────────────────────────────
# 4. 【调整后】在缩小后的样本上做 dark kitchen 排查
#    —— 现在样本量应该跟小边界那次(704条量级)接近，人工核查才可行
# ────────────────────────────────────────────────────────────
addr = (df_food_r1900["AddressLine1"].fillna("") + " " + df_food_r1900.get("AddressLine2", "").fillna(""))

suspects_tight = df_food_r1900[addr.str.contains(
    r"\bUnit\b|\bArch\b|\bRear\b|Industrial|Estate|Warehouse",
    case=False, na=False, regex=True
)]
other = df_food_r1900[df_food_r1900["BusinessType"] == "Other catering premises"]

suspects_tight[["BusinessName", "AddressLine1", "AddressLine2", "BusinessType"]].to_csv(
    f"{DATA_DIR}/_review_suspects_tight.csv", index=False
)
other[["BusinessName", "AddressLine1", "AddressLine2"]].to_csv(
    f"{DATA_DIR}/_review_other_catering.csv", index=False
)

print(f"半径筛选后疑似dark kitchen数: {len(suspects_tight)}  →  已存csv")
print(f"半径筛选后Other catering premises数: {len(other)}  →  已存csv")
print()

# ────────────────────────────────────────────────────────────
# 5. 人工排除（你需要重新核对上面两份新csv，清单大概率还是要更新，
#    但这次样本量应该回到可控范围，不会再出现600+条机构餐饮）
# ────────────────────────────────────────────────────────────
exclude_names = ["Vacherin", "Fooditude", "Duolingo", "Pinterest", "Burberry", "Marshall Street"]
exclude_mask = df_food_r1900["BusinessName"].str.contains("|".join(exclude_names), case=False, na=False)
print(f"按（可能需要更新的）排除清单匹配到: {exclude_mask.sum()} 条")
df_final = df_food_r1900[~exclude_mask].copy()

print(f"\n========== 关键结果汇总 ==========")
print(f"最终POI数: {len(df_final)}")
for bt, cnt in df_final['BusinessType'].value_counts().items():
    print(f"  {bt}: {cnt}")

cols = ["BusinessName", "BusinessType", "AddressLine1", "PostCode", lon_col, lat_col, "RatingValue", "FHRSID"]
cols = [c for c in cols if c in df_final.columns]
df_final[cols].to_csv(OUTPUT_PATH, index=False)
print(f"存为: {OUTPUT_PATH}")

宽域原始记录数: 16528
BusinessType
Restaurant/Cafe/Canteen                  8531
Retailers - other                        2724
Takeaway/sandwich shop                   1840
Pub/bar/nightclub                        1017
Other catering premises                   612
Hotel/bed & breakfast/guest house         525
Retailers - supermarkets/hypermarkets     345
School/college/university                 310
Caring Premises                           208
Manufacturers/packers                     156
Mobile caterer                            110
Distributors/Transporters                  84
Importers/Exporters                        66
Name: count, dtype: int64

业态过滤后记录数: 10983
BusinessType
Restaurant/Cafe/Canteen    8531
Takeaway/sandwich shop     1840
Other catering premises     612
Name: count, dtype: int64

半径筛选后（Soho r=1900m范围内、业态过滤后）记录数: 3714
BusinessType
Restaurant/Cafe/Canteen    3230
Takeaway/sandwich shop      333
Other catering premises     151
Name: count, dtype: int64

半径筛选后疑似dark kitchen数:

In [4]:
# ────────────────────────────────────────────────────────────
# 5. 规则式排除（替代原来只有6个名字的排除清单）
#    每条规则对应一个可在方法论里说清楚的判断依据
# ────────────────────────────────────────────────────────────

# 规则1：已知的企业合同餐饮服务商（这些公司专门做办公室/机构内部
# 供餐，不面向消费者，也不产生外卖订单）
contract_caterers = [
    "Bartlett Mitchell", "BaxterStorey", "Baxter Storey", "Restaurant Associates",
    "Vacherin", "Compass Group", "Gather & Gather", "Gather And Gather",
    "Searcys", "Graysons", "Eurest", "ISS Facility", "ISS Facilities",
    "Company of Cooks", "Blue Apple Catering", "Sodexo",
]

# 规则2：泰晤士河游船/派对船（地址含"MV "前缀）
is_boat = df_food_r1900["BusinessName"].str.match(r"^MV\s", case=False, na=False)

# 规则3：宗教/慈善/社区机构
institutional_keywords = [
    "Church", "Chapel", "Synagogue", "Temple", "Salvation Army",
    "Community Centre", "Youth Centre", "Youth Club", "Hostel",
    "Society", "Charity", "Trust"
]

# 规则4：非餐饮场馆误分类
venue_keywords = [
    "Theatre", "Cinema", "Museum", "Casino", "Conference Centre", "Hospital"
]

# 规则5：之前已确认需排除的具体商户
known_exclusions = ["Fooditude", "Duolingo", "Pinterest", "Burberry"]

all_exclusion_patterns = contract_caterers + institutional_keywords + venue_keywords + known_exclusions
name_or_addr = df_food_r1900["BusinessName"].fillna("") + " " + df_food_r1900["AddressLine1"].fillna("")

exclude_mask = (
    name_or_addr.str.contains("|".join(all_exclusion_patterns), case=False, na=False, regex=True)
    | is_boat
)

print(f"规则式排除匹配到: {exclude_mask.sum()} 条")
df_final = df_food_r1900[~exclude_mask].copy()

规则式排除匹配到: 242 条


In [5]:
# 单独看一下这次排除命中的、但原本不属于Other catering premises的记录
extra_hits = df_food_r1900[exclude_mask & (df_food_r1900["BusinessType"] != "Other catering premises")]
print(f"命中排除规则、但业态是Restaurant/Takeaway的记录数: {len(extra_hits)}")
print(extra_hits[["BusinessName", "AddressLine1", "BusinessType"]].to_string())

命中排除规则、但业态是Restaurant/Takeaway的记录数: 164
                                                                     BusinessName                                                     AddressLine1             BusinessType
35                                                                Adelphi Theatre                                           GATTI HOUSE 410 STRAND  Restaurant/Cafe/Canteen
53                                                                Aldwych Theatre                                                       49 ALDWYCH  Restaurant/Cafe/Canteen
87                                                                 Apollo Theatre                          APOLLO THEATRE 31-33 SHAFTESBURY AVENUE  Restaurant/Cafe/Canteen
132                                                             Autograph At UCLH                              University College London  Hospital  Restaurant/Cafe/Canteen
149                                                     BAD BOY PIZZA SOCIETY LTD                   

In [6]:
# 单独检查 Society / Trust 这两个高风险关键词各自命中了哪些记录，
# 存成csv，不在屏幕上整段打印，避免再被截断
risky_keywords = ["Society", "Trust"]
risky_mask = name_or_addr.str.contains("|".join(risky_keywords), case=False, na=False, regex=True)
risky_hits = df_food_r1900[risky_mask]

risky_hits[["BusinessName", "AddressLine1", "BusinessType"]].to_csv(
    f"{DATA_DIR}/_review_society_trust_hits.csv", index=False
)
print(f"Society/Trust 关键词命中: {len(risky_hits)} 条 → 已存 _review_society_trust_hits.csv")

Society/Trust 关键词命中: 19 条 → 已存 _review_society_trust_hits.csv


In [7]:
# Society/Trust 关键词误伤的真实商户，手动确认后从排除范围里摘出来保留
whitelist_names = [
    "BAD BOY PIZZA SOCIETY LTD",
    "Cellar Society",
    "SOFT SERVE SOCIETY",
    "Soft Serve Society",
    "Mr Fogg's Society Of Exploration",
]
is_whitelisted = df_food_r1900["BusinessName"].isin(whitelist_names)

# 保留原来的规则式排除，但在最后把白名单命中的记录从排除范围里摘出来
exclude_mask_final = exclude_mask & ~is_whitelisted

print(f"原规则式排除: {exclude_mask.sum()} 条")
print(f"白名单摘出: {is_whitelisted.sum()} 条")
print(f"最终排除: {exclude_mask_final.sum()} 条")

df_final = df_food_r1900[~exclude_mask_final].copy()
print(f"\n========== 关键结果汇总 ==========")
print(f"最终POI数: {len(df_final)}")
for bt, cnt in df_final['BusinessType'].value_counts().items():
    print(f"  {bt}: {cnt}")

cols = ["BusinessName", "BusinessType", "AddressLine1", "PostCode", lon_col, lat_col, "RatingValue", "FHRSID"]
cols = [c for c in cols if c in df_final.columns]
df_final[cols].to_csv(OUTPUT_PATH, index=False)
print(f"存为: {OUTPUT_PATH}")

原规则式排除: 242 条
白名单摘出: 5 条
最终排除: 237 条

========== 关键结果汇总 ==========
最终POI数: 3477
  Restaurant/Cafe/Canteen: 3077
  Takeaway/sandwich shop: 327
  Other catering premises: 73
存为: /Users/wusiyi/Documents/CASA/dis/DATA/soho_food_pois_r1900_clean.csv


In [9]:
import sys
sys.path.insert(0, "/Users/wusiyi/Documents/CASA/dis/DATA/files")
from environment import load_environment
env = load_environment("/Users/wusiyi/Documents/CASA/dis/DATA")

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 5313，路段 11210
  [env] 路段宽度：224 条来自 OSM width 标签，其余用类型默认值
  [env] POI 3477 个吸附到 1340 个节点
  [env] land_use_mix：均值 0.09，最大 1.00
  [env] activity landuse：604 个地块（类别：['commercial', 'education', 'governmental', 'religious', 'residential', 'retail']）
  [env] activity_density：均值 0.05，最大 1.00
  [env] 装卸区 39 个吸附到 24 个节点
  [env] 自行车停放 1523 个吸附到 915 个节点
  [env] 边界已加载并投影到 EPSG:32630


In [10]:
import numpy as np

vals = np.array(list(env.land_use_mix.values()))
print(f"land_use_mix 分布: 均值={vals.mean():.4f}, 中位数={np.median(vals):.4f}")
print(f"当前 land_use_threshold=0.20 在新分布中的百分位: {(vals < 0.20).mean()*100:.2f}%")
print(f"新分布的 95/99/99.5 百分位数值: {np.percentile(vals, [95, 99, 99.5])}")

land_use_mix 分布: 均值=0.0924, 中位数=0.0370
当前 land_use_threshold=0.20 在新分布中的百分位: 85.36%
新分布的 95/99/99.5 百分位数值: [0.36820765 0.6548516  0.77752013]


In [12]:
from model import SohoDeliveryModel

n_steps = int(60 * 60 / 5)  # 60分钟短程测试，dt=5秒 → 720步
candidates = [0.35, 0.65, 0.78]  # 对应新分布的95/99/99.5百分位

for threshold in candidates:
    model = SohoDeliveryModel(
        env=env,
        scenario="S1_baseline",
        seed=42,
        land_use_threshold=threshold,
    )
    for _ in range(n_steps):
        model.step()

    diag = model.quadrant_diagnostics()
    print(f"threshold={threshold}:")
    for k, v in diag.items():
        print(f"    {k}: {v}")
    print()

/Users/wusiyi/Documents/CASA/dis/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


threshold=0.35:
    full_compliance: 0.5125191619826265
    forced_minor_violation: 0.01777502007445799
    active_violation: 0.44382801664355065
    combined_high_risk: 0.025877801299364917
    _congestion_p99: 0.75
    _degenerate: True

threshold=0.65:
    full_compliance: 0.5211694284254326
    forced_minor_violation: 0.013395138331265056
    active_violation: 0.4432075333965983
    combined_high_risk: 0.02222789984670414
    _congestion_p99: 0.6666666666666666
    _degenerate: True

threshold=0.78:
    full_compliance: 0.5200379589751076
    forced_minor_violation: 0.015001094970435799
    active_violation: 0.4385356595371925
    combined_high_risk: 0.026425286517264033
    _congestion_p99: 0.75
    _degenerate: True



In [13]:
# 第一步：先看land_use_threshold往更低走会不会有变化
# （之前只测了0.35-0.78，都在新分布的95百分位以上；
#  试试更低的值，看象限占比是否真的对这个参数敏感）
lower_candidates = [0.05, 0.10, 0.15, 0.037]  # 0.037是新分布的中位数

for threshold in lower_candidates:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42, land_use_threshold=threshold)
    for _ in range(n_steps):
        model.step()
    diag = model.quadrant_diagnostics()
    print(f"land_use_threshold={threshold}: forced_minor={diag.get('forced_minor_violation'):.4f}, "
          f"combined_high={diag.get('combined_high_risk'):.4f}, degenerate={diag.get('_degenerate')}")

print()

# 第二步：如果上面还是没反应，测试land_use_weight本身
# （固定land_use_threshold=0.65，即之前测试里表现相对居中的一个）
for weight in [0.1, 0.2, 0.3, 0.5]:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42,
                               land_use_threshold=0.65, land_use_weight=weight)
    for _ in range(n_steps):
        model.step()
    diag = model.quadrant_diagnostics()
    print(f"land_use_weight={weight}: forced_minor={diag.get('forced_minor_violation'):.4f}, "
          f"combined_high={diag.get('combined_high_risk'):.4f}, degenerate={diag.get('_degenerate')}")

land_use_threshold=0.05: forced_minor=0.0165, combined_high=0.0358, degenerate=True
land_use_threshold=0.1: forced_minor=0.0124, combined_high=0.0174, degenerate=True
land_use_threshold=0.15: forced_minor=0.0118, combined_high=0.0197, degenerate=True
land_use_threshold=0.037: forced_minor=0.0141, combined_high=0.0172, degenerate=True

land_use_weight=0.1: forced_minor=0.0143, combined_high=0.0201, degenerate=True
land_use_weight=0.2: forced_minor=0.0137, combined_high=0.0211, degenerate=True
land_use_weight=0.3: forced_minor=0.0134, combined_high=0.0222, degenerate=True
land_use_weight=0.5: forced_minor=0.0164, combined_high=0.0306, degenerate=True


In [14]:
model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42)
for _ in range(n_steps):
    model.step()

df_agents = model.datacollector.get_agent_vars_dataframe()
print(df_agents["congestion_index"].describe(percentiles=[.25, .5, .75, .9, .95, .99]))
print(f"\n超过congestion_threshold=0.5的agent-step占比: {(df_agents['congestion_index'] > 0.5).mean():.4f}")

count    27398.000000
mean         0.196098
std          0.125872
min          0.000000
25%          0.125000
50%          0.153846
75%          0.200000
90%          0.333333
95%          0.375000
99%          0.750000
max          1.000000
Name: congestion_index, dtype: float64

超过congestion_threshold=0.5的agent-step占比: 0.0034


In [15]:
congestion_candidates = [0.15, 0.2, 0.25, 0.30]  # 大致对应现在分布的50-90百分位区间

for ct in congestion_candidates:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42,
                               congestion_threshold=ct)
    for _ in range(n_steps):
        model.step()
    diag = model.quadrant_diagnostics()
    print(f"congestion_threshold={ct}: "
          f"forced_minor={diag.get('forced_minor_violation'):.4f}, "
          f"combined_high={diag.get('combined_high_risk'):.4f}, "
          f"active={diag.get('active_violation'):.4f}, "
          f"compliance={diag.get('full_compliance'):.4f}, "
          f"degenerate={diag.get('_degenerate')}")

congestion_threshold=0.15: forced_minor=0.3425, combined_high=0.3036, active=0.1613, compliance=0.1925, degenerate=False
congestion_threshold=0.2: forced_minor=0.1837, combined_high=0.1749, active=0.2901, compliance=0.3513, degenerate=False
congestion_threshold=0.25: forced_minor=0.1148, combined_high=0.1252, active=0.3398, compliance=0.4203, degenerate=False
congestion_threshold=0.3: forced_minor=0.0608, combined_high=0.0780, active=0.3870, compliance=0.4742, degenerate=False


In [16]:
land_use_candidates = [0.35, 0.65, 0.78]  # 沿用同样的候选值，但这次固定congestion_threshold=0.2

for lut in land_use_candidates:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42,
                               congestion_threshold=0.2, land_use_threshold=lut)
    for _ in range(n_steps):
        model.step()
    diag = model.quadrant_diagnostics()
    print(f"land_use_threshold={lut} (congestion_threshold=0.2 固定): "
          f"forced_minor={diag.get('forced_minor_violation'):.4f}, "
          f"combined_high={diag.get('combined_high_risk'):.4f}, "
          f"active={diag.get('active_violation'):.4f}, "
          f"compliance={diag.get('full_compliance'):.4f}, "
          f"degenerate={diag.get('_degenerate')}")

land_use_threshold=0.35 (congestion_threshold=0.2 固定): forced_minor=0.1842, combined_high=0.1750, active=0.2947, compliance=0.3461, degenerate=False
land_use_threshold=0.65 (congestion_threshold=0.2 固定): forced_minor=0.1795, combined_high=0.1719, active=0.2936, compliance=0.3551, degenerate=False
land_use_threshold=0.78 (congestion_threshold=0.2 固定): forced_minor=0.1837, combined_high=0.1749, active=0.2901, compliance=0.3513, degenerate=False


In [17]:
land_use_candidates = [0.35, 0.65, 0.78]

for lut in land_use_candidates:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=42,
                               congestion_threshold=0.2, land_use_threshold=lut)
    for _ in range(n_steps):
        model.step()

    df_agents = model.datacollector.get_agent_vars_dataframe()
    route_share = df_agents["route_decision"].value_counts(normalize=True)
    print(f"land_use_threshold={lut}:")
    print(route_share.to_string())
    print()

land_use_threshold=0.35:
route_decision
compliant            0.722607
shortcut_pavement    0.277393

land_use_threshold=0.65:
route_decision
compliant            0.779692
shortcut_pavement    0.220308

land_use_threshold=0.78:
route_decision
compliant            0.785532
shortcut_pavement    0.214468



In [18]:
for seed in [42, 1, 7, 99, 123]:
    model = SohoDeliveryModel(env=env, scenario="S1_baseline", seed=seed,
                               congestion_threshold=0.2, land_use_threshold=0.65)
    for _ in range(n_steps):
        model.step()
    diag = model.quadrant_diagnostics()
    print(f"seed={seed}: forced_minor={diag.get('forced_minor_violation'):.4f}, "
          f"combined_high={diag.get('combined_high_risk'):.4f}, "
          f"degenerate={diag.get('_degenerate')}")

seed=42: forced_minor=0.1795, combined_high=0.1719, degenerate=False
seed=1: forced_minor=0.1696, combined_high=0.1597, degenerate=False
seed=7: forced_minor=0.1711, combined_high=0.1751, degenerate=False
seed=99: forced_minor=0.1897, combined_high=0.1691, degenerate=False
seed=123: forced_minor=0.1828, combined_high=0.1767, degenerate=False


In [19]:
n_severe_events = sum(1 for e in model.conflict_events if e["kind"] == "severe_pavement_conflict")
n_severe_from_deliveries = sum(d["severe_conflicts"] for d in model.completed_deliveries)

print(f"事件流水账里的severe数: {n_severe_events}")
print(f"已完成配送汇总的severe数: {n_severe_from_deliveries}")
print(f"差值: {n_severe_events - n_severe_from_deliveries}")

from agents import RiderAgent
unfinished_riders = [a for a in model.agents_by_type[RiderAgent] if not a.finished]
# 这些在途骑手当前这一单已经累积、但还没被计入completed_deliveries的severe_conflicts
in_flight_severe = sum(a.severe_conflicts for a in unfinished_riders)
print(f"仍在途骑手数: {len(unfinished_riders)}")
print(f"这些骑手当前这单已累积但未计入总和的severe_conflicts: {in_flight_severe}")

事件流水账里的severe数: 59
已完成配送汇总的severe数: 59
差值: 0
仍在途骑手数: 40
这些骑手当前这单已累积但未计入总和的severe_conflicts: 0


In [20]:
import sys
sys.path.insert(0, "/Users/wusiyi/Documents/CASA/dis/DATA/files")

from environment import load_environment
from model import SohoDeliveryModel

DATA_DIR = "/Users/wusiyi/Documents/CASA/dis/DATA"
env = load_environment(DATA_DIR)

n_steps = int(180 * 60 / 5)

model = SohoDeliveryModel(
    env=env,
    n_riders=300,
    n_pedestrians=600,
    seed=42,
    scenario="S1_baseline",
    congestion_threshold=0.3,
    land_use_threshold=0.65,
    jam_density=1.0,  # 当前默认值
)

for i in range(n_steps):
    model.step()
    if (i + 1) % max(n_steps // 10, 1) == 0:
        print(f"  {100 * (i + 1) // n_steps:3d}%")

df_agents = model.datacollector.get_agent_vars_dataframe()
ci = df_agents["congestion_index"]

print("\n========== congestion_index 分布（jam_density=1.0） ==========")
print(ci.describe(percentiles=[.5, .75, .9, .95, .99]))
print(f"\n达到饱和值1.0的agent-step占比: {(ci >= 1.0).mean():.4f}")

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 5313，路段 11210
  [env] 路段宽度：224 条来自 OSM width 标签，其余用类型默认值
  [env] POI 3477 个吸附到 1340 个节点
  [env] land_use_mix：均值 0.09，最大 1.00
  [env] activity landuse：604 个地块（类别：['commercial', 'education', 'governmental', 'religious', 'residential', 'retail']）
  [env] activity_density：均值 0.05，最大 1.00
  [env] 装卸区 39 个吸附到 24 个节点
  [env] 自行车停放 1523 个吸附到 915 个节点
  [env] 边界已加载并投影到 EPSG:32630
   10%
   20%
   30%
   40%
   50%
   60%
   70%
   80%
   90%
  100%

========== congestion_index 分布（jam_density=1.0） ==========
count    639656.000000
mean          0.293380
std           0.237343
min           0.000000
50%           0.181818
75%           0.363636
90%           0.625000
95%           1.000000
99%           1.000000
max           1.000000
Name: congestion_index, dtype: float64

达到饱和值1.0的agent-step占比: 0.0207


In [21]:
print("\n========== congestion_index 分布（jam_density=1.0） ==========")
desc = ci.describe(percentiles=[.5, .75, .9, .95, .99])
for k, v in desc.items():
    print(f"{k}: {v:.4f}")

print(f"\n达到饱和值1.0的agent-step占比: {(ci >= 1.0).mean():.4f}")


========== congestion_index 分布（jam_density=1.0） ==========
count: 639656.0000
mean: 0.2934
std: 0.2373
min: 0.0000
50%: 0.1818
75%: 0.3636
90%: 0.6250
95%: 1.0000
99%: 1.0000
max: 1.0000

达到饱和值1.0的agent-step占比: 0.0207


In [22]:
print("\n========== 精细分布（90-99百分位区间） ==========")
for p in [0.90, 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98, 0.99]:
    val = ci.quantile(p)
    print(f"{p:.2f}: {val:.6f}")

print(f"\n严格大于0.9的占比: {(ci > 0.9).mean():.4f}")
print(f"严格大于0.95的占比: {(ci > 0.95).mean():.4f}")
print(f"严格大于0.99的占比: {(ci > 0.99).mean():.4f}")
print(f"严格等于1.0的占比: {(ci == 1.0).mean():.4f}")


========== 精细分布（90-99百分位区间） ==========
0.90: 0.625000
0.91: 0.666667
0.92: 0.750000
0.93: 0.875000
0.94: 1.000000
0.95: 1.000000
0.96: 1.000000
0.97: 1.000000
0.98: 1.000000
0.99: 1.000000

严格大于0.9的占比: 0.0217
严格大于0.95的占比: 0.0207
严格大于0.99的占比: 0.0207
严格等于1.0的占比: 0.0207


In [25]:
# 找出所有congestion_index恰好等于1.0的记录，看它们分布在哪些edge上
sat_records = df_agents[df_agents["congestion_index"] == 1.0]
print(f"饱和记录数: {len(sat_records)}")

if "edge" in sat_records.columns:
    edge_counts = sat_records["edge"].value_counts()
    print(f"\n涉及不同edge数: {edge_counts.nunique()}")
    print(f"前10条边的饱和次数:")
    print(edge_counts.head(10))

饱和记录数: 39148


In [26]:
print(f"\n涉及不同edge数: {edge_counts.nunique()}")
print(f"前10条边的饱和次数:")
print(edge_counts.head(10))

NameError: name 'edge_counts' is not defined

In [27]:
print(df_agents.columns.tolist())

['stress_level', 'congestion_index', 'route_decision', 'route_trigger', 'baseline_tendency']


In [28]:
import sys
sys.path.insert(0, "/Users/wusiyi/Documents/CASA/dis/DATA/files")

from environment import load_environment
from model import SohoDeliveryModel
from agents import RiderAgent
from collections import Counter

DATA_DIR = "/Users/wusiyi/Documents/CASA/dis/DATA"
env = load_environment(DATA_DIR)

model = SohoDeliveryModel(
    env=env,
    n_riders=300,
    n_pedestrians=600,
    seed=42,
    scenario="S1_baseline",
    congestion_threshold=0.3,
    land_use_threshold=0.65,
)

n_steps = int(180 * 60 / 5)
edge_saturation_counter = Counter()   # 记录每条edge出现"饱和"的次数
edge_total_counter = Counter()        # 记录每条edge被观测到的总次数（分母，方便算饱和率）

for i in range(n_steps):
    model.step()
    for a in model.agents_by_type[RiderAgent]:
        edge = getattr(a, "current_edge", None)
        ci = getattr(a, "congestion_index", None)
        if edge is not None and ci is not None:
            edge_total_counter[edge] += 1
            if ci >= 1.0:
                edge_saturation_counter[edge] += 1
    if (i + 1) % max(n_steps // 10, 1) == 0:
        print(f"  {100 * (i + 1) // n_steps:3d}%")

print(f"\n涉及饱和的不同edge数: {len(edge_saturation_counter)}")
print(f"模型里总共有多少条edge被访问过: {len(edge_total_counter)}")
print(f"\n饱和次数最多的10条edge:")
for edge, count in edge_saturation_counter.most_common(10):
    total = edge_total_counter[edge]
    print(f"  {edge}: 饱和{count}次 / 共观测{total}次 (饱和率{count/total:.1%})")

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 5313，路段 11210
  [env] 路段宽度：224 条来自 OSM width 标签，其余用类型默认值
  [env] POI 3477 个吸附到 1340 个节点
  [env] land_use_mix：均值 0.09，最大 1.00
  [env] activity landuse：604 个地块（类别：['commercial', 'education', 'governmental', 'religious', 'residential', 'retail']）
  [env] activity_density：均值 0.05，最大 1.00
  [env] 装卸区 39 个吸附到 24 个节点
  [env] 自行车停放 1523 个吸附到 915 个节点
  [env] 边界已加载并投影到 EPSG:32630
   10%
   20%
   30%
   40%
   50%
   60%
   70%
   80%
   90%
  100%

涉及饱和的不同edge数: 203
模型里总共有多少条edge被访问过: 7274

饱和次数最多的10条edge:
  (3764074072, 8742067136): 饱和12961次 / 共观测14286次 (饱和率90.7%)
  (361242661, 109577): 饱和7514次 / 共观测9124次 (饱和率82.4%)
  (107818, 25257616): 饱和5574次 / 共观测9057次 (饱和率61.5%)
  (338748184, 33674190): 饱和1903次 / 共观测5025次 (饱和率37.9%)
  (26699565, 25376002): 饱和1858次 / 共观测2789次 (饱和率66.6%)
  (60852813, 235370948): 饱和1105次 / 共观测1387次 (饱和率79.7%)
  (235370948, 60852813): 饱和852次 / 共观测1548次 (饱和率55.0%)
  (5310577897, 8544994480): 饱和573次 / 共观测586次 (饱和率97.8%)
  (10313555617, 4844

In [29]:
top_edges = [e for e, _ in edge_saturation_counter.most_common(20)]

print("饱和率最高的边，对应的道路宽度：")
for edge in top_edges:
    u, v = edge
    if model.env.graph.has_edge(u, v):
        data = model.env.graph.get_edge_data(u, v)
        # 处理MultiDiGraph可能有多条平行边的情况
        if isinstance(data, dict) and 0 in data:
            data = data[0]
        width = data.get("carriageway_width", "未知")
        highway = data.get("highway", "未知")
        count = edge_saturation_counter[edge]
        total = edge_total_counter[edge]
        print(f"  edge={edge}: width={width}, highway={highway}, "
              f"饱和{count}/{total} ({count/total:.1%})")

饱和率最高的边，对应的道路宽度：
  edge=(3764074072, 8742067136): width=3.0, highway=cycleway, 饱和12961/14286 (90.7%)
  edge=(361242661, 109577): width=5.5, highway=['trunk_link', 'primary'], 饱和7514/9124 (82.4%)
  edge=(107818, 25257616): width=8.0, highway=primary, 饱和5574/9057 (61.5%)
  edge=(338748184, 33674190): width=8.0, highway=primary, 饱和1903/5025 (37.9%)
  edge=(26699565, 25376002): width=3.0, highway=cycleway, 饱和1858/2789 (66.6%)
  edge=(60852813, 235370948): width=3.0, highway=cycleway, 饱和1105/1387 (79.7%)
  edge=(235370948, 60852813): width=3.0, highway=cycleway, 饱和852/1548 (55.0%)
  edge=(5310577897, 8544994480): width=1.0, highway=cycleway, 饱和573/586 (97.8%)
  edge=(10313555617, 4844748294): width=3.0, highway=cycleway, 饱和462/775 (59.6%)
  edge=(11729098679, 1130845644): width=5.0, highway=cycleway, 饱和406/1097 (37.0%)
  edge=(9791495, 108239): width=8.0, highway=primary, 饱和317/2533 (12.5%)
  edge=(5453319326, 9512926): width=6.5, highway=tertiary, 饱和270/2792 (9.7%)
  edge=(7261577394, 2541

In [30]:
import osmnx as ox

target_edge = (107818, 25257616)
u, v = target_edge
data = model.env.graph.get_edge_data(u, v)
if isinstance(data, dict) and 0 in data:
    data = data[0]
print(data)

# 反投影回经纬度，方便你对照实际地图确认这是哪条路
node_u = model.env.graph.nodes[u]
node_v = model.env.graph.nodes[v]
print(f"起点坐标: {node_u.get('x')}, {node_u.get('y')}")
print(f"终点坐标: {node_v.get('x')}, {node_v.get('y')}")

{'osmid': [1135566097, 1135566098, 1135565524, 1135565525, 211486426], 'highway': 'primary', 'maxspeed': '20 mph', 'name': 'Piccadilly', 'oneway': True, 'reversed': False, 'length': 310.8253868677218, 'geometry': <LINESTRING (698822.292 5710403.871, 698820.175 5710399.535, 698817.694 5710...>, 'lanes': ['1', '2'], 'ref': 'A4', 'carriageway_width': 8.0, 'footway_width': 3.5}
起点坐标: 698822.2915322346, 5710403.871190917
终点坐标: 698555.156562099, 5710245.815672801
